In [ ]:
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests
from scipy.linalg import solve

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns                                  # add
from matplotlib.ticker import FormatStrFormatter       # add
from scipy.stats import pearsonr

sns.set_style('whitegrid')                             # add  → thin, light spines
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11})  # add


### Data Loading

Load precomputed trophic coherence and effective connectivity (Ceff) matrices from `.mat` files. Separate files exist for LOW and HIGH anhedonia groups. The comparison file contains both groups' trophic coherence scalars and Ceff matrices; per-group files additionally contain model fit metrics (fittFC, fittCVtau) and per-region hierarchical levels (NSUB × 232).

In [ ]:
# %% Load results from .mat files
# The comparison file has trophic coherence + Ceff
data_comp = sio.loadmat('outputs/NEW_4_factor_clustering_ipnybV4/results_Ceff_anhedonia_comparison.mat')
trophiccoherence_LOW = data_comp['trophiccoherence_LOW'].flatten()
trophiccoherence_HIGH = data_comp['trophiccoherence_HIGH'].flatten()

# Per-group files have fittFC, fittCVtau, hierarchicallevels
data_low = sio.loadmat('outputs/NEW_4_factor_clustering_ipnybV4/low_anhedonia/results_Ceff_low_anhedonia.mat')
data_high = sio.loadmat('outputs/NEW_4_factor_clustering_ipnybV4/high_anhedonia/results_Ceff_high_anhedonia.mat')
Ceff_LOW  = data_low['Ceff_LOW']    # (NSUB_LOW, 232, 232)
Ceff_HIGH = data_high['Ceff_HIGH']  # (NSUB_HIGH, 232, 232)



In [ ]:
fittFC_LOW = data_low['fittFC_LOW'].flatten()
fittFC_HIGH = data_high['fittFC_HIGH'].flatten()
fittCVtau_LOW = data_low['fittCVtau_LOW'].flatten()
fittCVtau_HIGH = data_high['fittCVtau_HIGH'].flatten()
hierarchicallevels_LOW = data_low['hierarchicallevels_LOW']   # (NSUB x 232)
hierarchicallevels_HIGH = data_high['hierarchicallevels_HIGH'] # (NSUB x 232)

NSUB_LOW = len(trophiccoherence_LOW)
NSUB_HIGH = len(trophiccoherence_HIGH)

In [ ]:
# %% Load region labels
label_file = '/Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/atlases/Schaefer2018_200Parcels_7Networks_order_Tian_Subcortex_S2_label.txt'

with open(label_file, 'r') as f:
    lines = f.read().strip().splitlines()

# Every odd line (0-indexed) is the label name, every even line is "index R G B A"
region_labels = [lines[i] for i in range(0, len(lines), 2)]

# Verify: should be 232 labels, and indices should match 1-232
assert len(region_labels) == 232, f'Expected 232 labels, got {len(region_labels)}'

# Now region_labels[i] is the name for column i of hierarchicallevels_LOW/HIGH
# e.g. region_labels[0] = 'aHIP-rh', region_labels[231] = '7Networks_RH_Default_pCunPCC_3'
print(region_labels[:5])   # first 5
print(region_labels[32:35]) # first cortical LH regions


### Defnining ROIs

In [ ]:
# %% Frontostriatal edge set — ROI index definitions (0-based, into Ceff/region_labels)
# Theory-driven sparse PFC<->striatum edge set for the information-flow asymmetry analysis.
# See anhedonia_circuits/frontostriatal_paper/frontostriatal_roi_interactions.md
import numpy as np

def _idx(pred):
    return [i for i, l in enumerate(region_labels) if pred(l)]

# --- Striatal nodes (Tian-S2) ---
NAc_idx  = [8, 9, 24, 25]   # NAc-shell/core rh+lh  (ventral striatum)
aPUT_idx = [12, 28]         # anterior putamen rh+lh (posterior pPUT [13,29] excluded = motor)

# --- Cortical nodes (Schaefer-200 7Networks), via the filter expressions in the md ---
mPFC_idx  = _idx(lambda l: ('Default_PFC' in l) and ('pCun' not in l))   # vmPFC/mPFC (21 parcels)
OFC_idx   = _idx(lambda l: ('Limbic_OFC' in l) or ('Cont_OFC' in l))     # OFC, med+lat merged (6)
ACC_idx   = _idx(lambda l: ('SalVentAttn_Med' in l) or ('Cont_Cing' in l))  # dACC proxy (10)
dlPFC_idx = _idx(lambda l: 'Cont_PFCl' in l)                             # dlPFC (12)

# --- Sanity checks against the documented index lists ---
assert NAc_idx  == [8, 9, 24, 25]
assert aPUT_idx == [12, 28]
assert mPFC_idx  == [114,115,116,117,118,119,120,121,122,123,124,125,126,221,222,223,224,225,226,227,228]
assert OFC_idx   == [86, 87, 96, 190, 191, 192]
assert ACC_idx   == [83, 84, 85, 103, 104, 187, 188, 189, 209, 210]
assert dlPFC_idx == [97, 98, 99, 100, 101, 201, 202, 203, 204, 205, 206, 207]

print("ROI sizes:",
      {k: len(v) for k, v in
       dict(NAc=NAc_idx, aPUT=aPUT_idx, mPFC=mPFC_idx,
            OFC=OFC_idx, ACC=ACC_idx, dlPFC=dlPFC_idx).items()})


### edge definitions + directed-flow extraction (respects Ceff[target, source])

In [ ]:
# %% Directed information flow on the six edges
# CONVENTION (from hopf_int.m: Jacobian Axx = a*I - diag(rowsum) + gC):
#   Ceff[i, j] is the directed edge  j -> i   (row = TARGET, column = SOURCE).
# Therefore for PFC node P and striatal node S:
#   F(P -> S)  (top-down)  = mean Ceff[S, P]   -> rows=striatal targets, cols=PFC sources
#   F(S -> P)  (bottom-up) = mean Ceff[P, S]
#   ASym = F(P->S) - F(S->P)   (positive = net TOP-DOWN, PFC drives striatum)

# Edge name -> (PFC index set, striatal index set). Names use '_' so they're valid
# as statsmodels formula column stems (e.g. 'vmPFC_NAc_asym ~ anhedonia_group + ...').
EDGES = {
    'vmPFC_NAc':  (mPFC_idx,  NAc_idx),   # 1  reward valuation core
    'OFC_NAc':    (OFC_idx,   NAc_idx),   # 2  reward valuation core
    'ACC_NAc':    (ACC_idx,   NAc_idx),   # 3  reward cingulate (dACC proxy)
    'dlPFC_NAc':  (dlPFC_idx, NAc_idx),   # 4  compensatory top-down
    'vmPFC_aPUT': (mPFC_idx,  aPUT_idx),  # 5  dorsal-striatal reward
    'ACC_aPUT':   (ACC_idx,   aPUT_idx),  # 6  dorsal-striatal reward
}

def directed_flow(Ceff_subj, pfc_idx, str_idx):
    """Return (F_topdown, F_bottomup) for one subject's 232x232 Ceff matrix.
       F_topdown  = mean F(PFC -> Str) = mean Ceff[str, pfc]
       F_bottomup = mean F(Str -> PFC) = mean Ceff[pfc, str]"""
    topdown  = Ceff_subj[np.ix_(str_idx, pfc_idx)].mean()  # PFC -> Str
    bottomup = Ceff_subj[np.ix_(pfc_idx, str_idx)].mean()  # Str -> PFC
    return topdown, bottomup

def edge_measures_for_group(Ceff_group):
    """Ceff_group: (NSUB, 232, 232). Returns dict[edge] -> dict of (NSUB,) arrays:
       'td' (P->S), 'bu' (S->P), 'asym' (td-bu), 'norm' ((td-bu)/(td+bu))."""
    nsub = Ceff_group.shape[0]
    out = {}
    for name, (pfc_idx, str_idx) in EDGES.items():
        td = np.empty(nsub); bu = np.empty(nsub)
        for s in range(nsub):
            td[s], bu[s] = directed_flow(Ceff_group[s], pfc_idx, str_idx)
        with np.errstate(divide='ignore', invalid='ignore'):
            norm = (td - bu) / (td + bu)
        out[name] = {'td': td, 'bu': bu, 'asym': td - bu, 'norm': norm}
    return out

edges_LOW  = edge_measures_for_group(Ceff_LOW)
edges_HIGH = edge_measures_for_group(Ceff_HIGH)

# Quick peek
for name in EDGES:
    print(f"{name:11s}  ASym  LOW={edges_LOW[name]['asym'].mean():+.4f}   "
          f"HIGH={edges_HIGH[name]['asym'].mean():+.4f}")


### assemble a per-subject dataframe merged with demographics, ready for OLS:

In [ ]:
# %% Subject IDs + demographics (needed for ids_low/ids_high and the OLS covariates)
import scipy.io as sio
import pandas as pd
import numpy as np

# --- Subject IDs per anhedonia group (order matches Ceff_LOW / Ceff_HIGH rows) ---
mat_low  = sio.loadmat('/Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/anhedonia/NEW_4_factor_clustering_ipnybV4/low_anhedonia.mat')
mat_high = sio.loadmat('/Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/anhedonia/NEW_4_factor_clustering_ipnybV4/high_anhedonia.mat')

ids_low  = [mat_low['low_anhedonia'][i, 1].flat[0]   for i in range(mat_low['low_anhedonia'].shape[0])]
ids_high = [mat_high['high_anhedonia'][i, 1].flat[0] for i in range(mat_high['high_anhedonia'].shape[0])]

# --- Demographics + derived covariate columns (Age_clean, is_patient) ---
demos = pd.read_excel(
    '/Users/proghani/Documents/personal/local_thesis_proj/TCP_thesis_project/my_analysis/trophic_coherence_analysis/data/demos_payman.xls',
    header=1
)
demos['Age_clean']  = demos['Age'].replace(999, np.nan)
demos['is_patient'] = demos['Primary_Dx_Payman'].apply(lambda x: 0 if x == 999 else 1)

print(f'N LOW  = {len(ids_low)}   (Ceff_LOW rows  = {Ceff_LOW.shape[0]})')
print(f'N HIGH = {len(ids_high)}  (Ceff_HIGH rows = {Ceff_HIGH.shape[0]})')
assert len(ids_low)  == Ceff_LOW.shape[0],  'ids_low length must match Ceff_LOW row count'
assert len(ids_high) == Ceff_HIGH.shape[0], 'ids_high length must match Ceff_HIGH row count'


In [ ]:
# %% Build per-subject frontostriatal dataframe (wide), ready for covariate-corrected OLS
# One row per subject; columns: <edge>_td, <edge>_bu, <edge>_asym, <edge>_norm for all 6 edges.
# Mirrors the OLS setup used elsewhere in the project (merge on subjectkey with demos;
# reference levels LOW / Female / HC).
import pandas as pd

def _group_frame(ids, edges_group, group_label):
    cols = {'subjectkey': ids, 'anhedonia_group': group_label}
    for name in EDGES:
        for m in ('td', 'bu', 'asym', 'norm'):
            cols[f'{name}_{m}'] = edges_group[name][m]
    return pd.DataFrame(cols)

df_fs = pd.concat([
    _group_frame(ids_low,  edges_LOW,  'LOW'),
    _group_frame(ids_high, edges_HIGH, 'HIGH'),
], ignore_index=True).merge(
    demos[['subjectkey', 'Age_clean', 'sex', 'is_patient']],
    on='subjectkey', how='left'
)

# Same recoding / reference levels as the other OLS models in the project
df_fs['sex'] = df_fs['sex'].replace('O', np.nan)
df_fs['anhedonia_group'] = pd.Categorical(df_fs['anhedonia_group'], categories=['LOW', 'HIGH'])
df_fs['sex']             = pd.Categorical(df_fs['sex'],             categories=['F', 'M'])

# Convenience lists for looping the OLS / FDR step later
EDGE_NAMES   = list(EDGES.keys())                       # 6 edges
ASYM_COLS    = [f'{e}_asym' for e in EDGE_NAMES]        # primary outcomes (top-down ASym)
RAW_TD_COLS  = [f'{e}_td'   for e in EDGE_NAMES]        # raw F(P->S)
RAW_BU_COLS  = [f'{e}_bu'   for e in EDGE_NAMES]        # raw F(S->P)
NORM_COLS    = [f'{e}_norm' for e in EDGE_NAMES]        # normalized index

print(df_fs.shape)
print(df_fs[['subjectkey', 'anhedonia_group'] + ASYM_COLS].head())


### OLS analyses

#### OFC<->NAc information-flow asymmetry

significant: vmPFC_NAc, dlPFC_NAc, vmPFC_aPUT, ACC_aPUT
not: OFC_NAc, ACC_NAc

In [ ]:
# %% Sensitivity: OFC<->NAc Asymmetry — Full / Patients Only / Patients + Broad Dx Category
import statsmodels.formula.api as smf
import pandas as pd
import numpy as np

# Broad diagnostic-category mapping (same as reference notebook)
dx_map = {
    'MDD': 'Mood', 'BP1': 'Mood', 'BP2': 'Mood',
    'Dysthymia': 'Mood', 'Other Mood Dis': 'Mood',
    'GAD': 'Anxiety_Trauma', 'PTSD': 'Anxiety_Trauma',
    'SAD': 'Anxiety_Trauma', 'Other Anxiety': 'Anxiety_Trauma',
    'SZ': 'Other', 'SZA': 'Other',
    'OCD': 'Other', 'SUD': 'Other',
    'ADHD': 'Other', 'ED': 'Other'
}

# df_fs doesn't carry Primary_Dx_Payman; merge it in and build the broad-Dx category
df_fs_sens = df_fs.merge(
    demos[['subjectkey', 'Primary_Dx_Payman']], on='subjectkey', how='left'
)

df_fs_sens['Dx_label'] = (
    df_fs_sens['Primary_Dx_Payman'].replace(999, 'HC').astype(str).str.strip()
)
df_fs_sens['Dx_broad'] = pd.Categorical(
    df_fs_sens['Dx_label'].map(dx_map),
    categories=['Mood', 'Anxiety_Trauma', 'Other']
)

OUTCOME = 'ACC_aPUT_norm'

# --- Model 1: Full sample (+ is_patient), refit for a clean comparison row ---
res_full = smf.ols(f'{OUTCOME} ~ anhedonia_group + Age_clean + sex + is_patient',
                   data=df_fs_sens).fit()
coef_full = res_full.params['anhedonia_group[T.HIGH]']
pval_full = res_full.pvalues['anhedonia_group[T.HIGH]']

# --- Model 2: Patients only (is_patient is constant -> dropped) ---
df_pat = df_fs_sens[df_fs_sens['is_patient'] == 1].copy()
n_low_pat  = (df_pat['anhedonia_group'] == 'LOW').sum()
n_high_pat = (df_pat['anhedonia_group'] == 'HIGH').sum()

res_pat = smf.ols(f'{OUTCOME} ~ anhedonia_group + Age_clean + sex', data=df_pat).fit()
coef_pat = res_pat.params['anhedonia_group[T.HIGH]']
pval_pat = res_pat.pvalues['anhedonia_group[T.HIGH]']
ci_lo_p, ci_hi_p = res_pat.conf_int().loc['anhedonia_group[T.HIGH]']

# --- Model 3: Patients only + broad diagnostic category ---
print('Dx_broad counts by group:')
print(pd.crosstab(df_pat['anhedonia_group'], df_pat['Dx_broad']))
print()

res_dx = smf.ols(f'{OUTCOME} ~ anhedonia_group + Age_clean + sex + Dx_broad',
                 data=df_pat.dropna(subset=['Dx_broad'])).fit()
coef_dx = res_dx.params['anhedonia_group[T.HIGH]']
pval_dx = res_dx.pvalues['anhedonia_group[T.HIGH]']
ci_lo_d, ci_hi_d = res_dx.conf_int().loc['anhedonia_group[T.HIGH]']

# --- Print results ---
ci_lo_f, ci_hi_f = res_full.conf_int().loc['anhedonia_group[T.HIGH]']
print('==============================')
print('OFC<->NAc ASYMMETRY — Full Sample (+ is_patient)')
print(f'  N = {int(res_full.nobs)}')
print('==============================')
print('  Anhedonia group (HIGH vs LOW):')
print(f'    β  = {coef_full:+.5f}')
print(f'    95% CI = [{ci_lo_f:+.5f}, {ci_hi_f:+.5f}]')
print(f'    p  = {pval_full:.4f}  {"*** significant" if pval_full < 0.05 else "ns"}')
print('\n  Covariates:')
for term in ['Age_clean', 'sex[T.M]', 'is_patient']:
    print(f'    {term:<22}  β={res_full.params[term]:+.5f},  p={res_full.pvalues[term]:.4f}')
print(f'\n  Model fit:  R²={res_full.rsquared:.3f},  adj-R²={res_full.rsquared_adj:.3f}\n')

print('==============================')
print('OFC<->NAc ASYMMETRY — Patients Only')
print(f'  N = {int(res_pat.nobs)}  (LOW={n_low_pat}, HIGH={n_high_pat})')
print('==============================')
print('  Anhedonia group (HIGH vs LOW):')
print(f'    β  = {coef_pat:+.5f}')
print(f'    95% CI = [{ci_lo_p:+.5f}, {ci_hi_p:+.5f}]')
print(f'    p  = {pval_pat:.4f}  {"*** significant" if pval_pat < 0.05 else "ns"}')
print('\n  Covariates:')
for term in ['Age_clean', 'sex[T.M]']:
    print(f'    {term:<22}  β={res_pat.params[term]:+.5f},  p={res_pat.pvalues[term]:.4f}')
print(f'\n  Model fit:  R²={res_pat.rsquared:.3f},  adj-R²={res_pat.rsquared_adj:.3f}')

print('\n==============================')
print('OFC<->NAc ASYMMETRY — Patients Only + Broad Dx Category')
print(f'  N = {int(res_dx.nobs)}')
print('==============================')
print('  Anhedonia group (HIGH vs LOW):')
print(f'    β  = {coef_dx:+.5f}')
print(f'    95% CI = [{ci_lo_d:+.5f}, {ci_hi_d:+.5f}]')
print(f'    p  = {pval_dx:.4f}  {"*** significant" if pval_dx < 0.05 else "ns"}')
print('\n  Covariates:')
for term in ['Age_clean', 'sex[T.M]', 'Dx_broad[T.Anxiety_Trauma]', 'Dx_broad[T.Other]']:
    print(f'    {term:<35}  β={res_dx.params[term]:+.5f},  p={res_dx.pvalues[term]:.4f}')
print(f'\n  Model fit:  R²={res_dx.rsquared:.3f},  adj-R²={res_dx.rsquared_adj:.3f}')

print('\n  ── Three-way comparison ──')
print(f'  {"Model":<40} {"β":>10} {"p":>9}')
print(f'  {"Full sample (+ is_patient)":<40} {coef_full:>+10.5f} {pval_full:>9.4f}')
print(f'  {"Patients only":<40} {coef_pat:>+10.5f} {pval_pat:>9.4f}')
print(f'  {"Patients + broad Dx category":<40} {coef_dx:>+10.5f} {pval_dx:>9.4f}')


In [ ]:
# %% Sensitivity (3 models) for all six edges — looped
import statsmodels.formula.api as smf
import pandas as pd
import numpy as np

# --- Build the sensitivity frame ONCE (Dx_broad added) ---
dx_map = {
    'MDD': 'Mood', 'BP1': 'Mood', 'BP2': 'Mood',
    'Dysthymia': 'Mood', 'Other Mood Dis': 'Mood',
    'GAD': 'Anxiety_Trauma', 'PTSD': 'Anxiety_Trauma',
    'SAD': 'Anxiety_Trauma', 'Other Anxiety': 'Anxiety_Trauma',
    'SZ': 'Other', 'SZA': 'Other', 'OCD': 'Other', 'SUD': 'Other',
    'ADHD': 'Other', 'ED': 'Other'
}
df_fs_sens = df_fs.merge(demos[['subjectkey', 'Primary_Dx_Payman']], on='subjectkey', how='left')
df_fs_sens['Dx_label'] = df_fs_sens['Primary_Dx_Payman'].replace(999, 'HC').astype(str).str.strip()
df_fs_sens['Dx_broad'] = pd.Categorical(df_fs_sens['Dx_label'].map(dx_map),
                                        categories=['Mood', 'Anxiety_Trauma', 'Other'])


def run_sensitivity(outcome, data, verbose=True):
    """Fit the 3 sensitivity models for one outcome column; return a results dict."""
    g = 'anhedonia_group[T.HIGH]'

    # Model 1: full sample (+ is_patient)
    res_full = smf.ols(f'{outcome} ~ anhedonia_group + Age_clean + sex + is_patient', data=data).fit()

    # Model 2: patients only (is_patient constant -> dropped)
    df_pat = data[data['is_patient'] == 1].copy()
    res_pat = smf.ols(f'{outcome} ~ anhedonia_group + Age_clean + sex', data=df_pat).fit()

    # Model 3: patients only + broad Dx
    res_dx = smf.ols(f'{outcome} ~ anhedonia_group + Age_clean + sex + Dx_broad',
                     data=df_pat.dropna(subset=['Dx_broad'])).fit()

    out = {
        'outcome': outcome,
        'beta_full': res_full.params[g], 'p_full': res_full.pvalues[g],
        'beta_pat':  res_pat.params[g],  'p_pat':  res_pat.pvalues[g],
        'beta_dx':   res_dx.params[g],   'p_dx':   res_dx.pvalues[g],
        'N_full': int(res_full.nobs), 'N_pat': int(res_pat.nobs), 'N_dx': int(res_dx.nobs),
    }

    if verbose:
        print(f'\n############### {outcome} ###############')
        print(f'  {"Model":<32}{"N":>5}{"β":>11}{"p":>9}')
        print(f'  {"Full (+is_patient)":<32}{out["N_full"]:>5}{out["beta_full"]:>+11.5f}{out["p_full"]:>9.4f}'
              f'  {"sig" if out["p_full"]<0.05 else "ns"}')
        print(f'  {"Patients only":<32}{out["N_pat"]:>5}{out["beta_pat"]:>+11.5f}{out["p_pat"]:>9.4f}'
              f'  {"sig" if out["p_pat"]<0.05 else "ns"}')
        print(f'  {"Patients + broad Dx":<32}{out["N_dx"]:>5}{out["beta_dx"]:>+11.5f}{out["p_dx"]:>9.4f}'
              f'  {"sig" if out["p_dx"]<0.05 else "ns"}')
    return out


# --- Loop over the six edges (pick the measure: 'asym', 'norm', 'td', 'bu') ---
MEASURE  = 'norm'
outcomes = [f'{e}_{MEASURE}' for e in EDGE_NAMES]   # EDGE_NAMES from Cell 3

sens_results = pd.DataFrame([run_sensitivity(o, df_fs_sens) for o in outcomes])
print('\n\n===== Summary table =====')
print(sens_results.to_string(index=False))


In [ ]:
from statsmodels.stats.multitest import multipletests

for col in ['p_full', 'p_pat', 'p_dx']:
    sens_results[col + '_FDR'] = multipletests(sens_results[col], method='fdr_bh')[1]

cols = ['outcome',
        'p_full', 'p_full_FDR',
        'p_pat',  'p_pat_FDR',
        'p_dx',   'p_dx_FDR']
print(sens_results[cols].to_string(index=False))


### Arm decomposition to understand which ways the information flow has changed 

In [ ]:
# %% Arm decomposition (top-down vs bottom-up) for edges significant in the FULL model (FDR)
import pandas as pd, numpy as np, statsmodels.formula.api as smf

def run_arm_ols(vals_low, vals_high, label):
    df = pd.concat([
        pd.DataFrame({'subjectkey': ids_low,  'flow': vals_low,  'anhedonia_group': 'LOW'}),
        pd.DataFrame({'subjectkey': ids_high, 'flow': vals_high, 'anhedonia_group': 'HIGH'}),
    ], ignore_index=True).merge(
        demos[['subjectkey', 'Age_clean', 'sex', 'is_patient']], on='subjectkey', how='left'
    )
    df['sex'] = df['sex'].replace('O', np.nan)
    df['anhedonia_group'] = pd.Categorical(df['anhedonia_group'], categories=['LOW', 'HIGH'])
    df['sex']             = pd.Categorical(df['sex'],             categories=['F', 'M'])

    result = smf.ols('flow ~ anhedonia_group + Age_clean + sex + is_patient', data=df).fit()
    coef = result.params['anhedonia_group[T.HIGH]']
    pval = result.pvalues['anhedonia_group[T.HIGH]']
    ci_lo, ci_hi = result.conf_int().loc['anhedonia_group[T.HIGH]']

    print('==============================')
    print(f'{label} — OLS Result')
    print('==============================')
    print(f'  Anhedonia group (HIGH vs LOW):')
    print(f'    β  = {coef:+.5f}   ({"HIGH > LOW (arm UP)" if coef > 0 else "HIGH < LOW (arm DOWN)"})')
    print(f'    95% CI = [{ci_lo:+.5f}, {ci_hi:+.5f}]')
    print(f'    p  = {pval:.4f}  {"*** significant" if pval < 0.05 else "ns"} (uncorrected)')
    print(f'\n  Model fit:  R²={result.rsquared:.3f},  N={int(result.nobs)}\n')
    return result, coef, pval

# --- Pick edges significant in the full model after FDR ---
sig_mask  = sens_results['p_full_FDR'] < 0.05
sig_edges = [o.replace(f'_{MEASURE}', '') for o in sens_results.loc[sig_mask, 'outcome']]
print(f'Edges significant in full model (FDR < .05): {sig_edges}\n')

# --- Decompose each significant edge into its two arms ---
for edge in sig_edges:
    print(f'\n################  {edge}  ################')
    _, b_td, p_td = run_arm_ols(edges_LOW[edge]['td'], edges_HIGH[edge]['td'],
                                f'{edge}  TOP-DOWN  F(PFC→Str)')
    _, b_bu, p_bu = run_arm_ols(edges_LOW[edge]['bu'], edges_HIGH[edge]['bu'],
                                f'{edge}  BOTTOM-UP F(Str→PFC)')

    print('  ── Arm direction summary (HIGH vs LOW) ──')
    print(f'    Top-down  F(PFC→Str):  β={b_td:+.5f}  p={p_td:.4f}')
    print(f'    Bottom-up F(Str→PFC):  β={b_bu:+.5f}  p={p_bu:.4f}')
    if b_td * b_bu < 0:
        verdict = 'OPPOSITE directions → congruent push, reinforces the asymmetry'
    else:
        verdict = 'SAME direction → asymmetry is a differential (one arm moved more)'
    print(f'    Arms moved in {verdict}\n')


### Correlations with SHAPS scores

In [ ]:
shaps_items = pd.read_csv('/Users/proghani/Documents/personal/local_thesis_proj/TCP_thesis_project/phenotype/imputed/imputed_shaps01.csv')
shaps_items['shaps_total']=shaps_items.iloc[:, 1:].sum(axis=1)


In [ ]:
# %% Correlation: per-edge asymmetry vs total SHAPS (whole sample, both groups pooled)
import pandas as pd, numpy as np
from scipy.stats import pearsonr, spearmanr

# --- SHAPS total (already built in your earlier cell) ---
# shaps_items['shaps_total'] = shaps_items.iloc[:, 1:].sum(axis=1)
# The first column of shaps_items is the subject ID — rename it to match df_fs['subjectkey'].
id_col = shaps_items.columns[0]
shaps = shaps_items[[id_col, 'shaps_total']].rename(columns={id_col: 'subjectkey'})

# --- Which measure + which edges ---
MEASURE   = 'norm'                 # same measure you tested ('norm'); use 'asym' for the raw difference
sig_edges = ['vmPFC_NAc', 'dlPFC_NAc', 'ACC_aPUT']   # the 3 full-model FDR survivors

# --- Merge edge values with SHAPS over the WHOLE sample ---
cols = ['subjectkey'] + [f'{e}_{MEASURE}' for e in sig_edges]
df_corr = df_fs[cols].merge(shaps, on='subjectkey', how='inner')

print(f'Merged rows: {len(df_corr)}  (df_fs={len(df_fs)}, shaps={len(shaps)})')
if len(df_corr) < min(len(df_fs), len(shaps)) * 0.8:
    print('  ⚠️  Few rows matched — check that the SHAPS ID column format matches subjectkey!')

# --- Correlate each edge's asymmetry with shaps_total ---
print(f'\nCorrelation of {MEASURE} asymmetry vs SHAPS total (whole sample):')
print(f'  {"edge":<12}{"n":>5}{"Pearson r":>12}{"p":>9}{"Spearman r":>13}{"p":>9}')
for e in sig_edges:
    col = f'{e}_{MEASURE}'
    sub = df_corr[[col, 'shaps_total']].dropna()
    r_p, p_p = pearsonr(sub[col], sub['shaps_total'])
    r_s, p_s = spearmanr(sub[col], sub['shaps_total'])
    print(f'  {e:<12}{len(sub):>5}{r_p:>+12.3f}{p_p:>9.4f}{r_s:>+13.3f}{p_s:>9.4f}')


In [ ]:
# %% Scatter plots: per-edge asymmetry vs total SHAPS (whole sample)
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter
from scipy.stats import pearsonr

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11})

MEASURE   = 'norm'
sig_edges = ['vmPFC_NAc', 'dlPFC_NAc', 'ACC_aPUT']
GROUP_COLORS = {'LOW': '#3AAFB9', 'HIGH': '#D65A7A'}

# Bring group label along for coloring (fit line still uses the whole sample)
plot_cols = ['subjectkey', 'anhedonia_group'] + [f'{e}_{MEASURE}' for e in sig_edges]
df_plot = df_fs[plot_cols].merge(shaps, on='subjectkey', how='inner')

fig, axes = plt.subplots(1, len(sig_edges), figsize=(5 * len(sig_edges), 4.5))
if len(sig_edges) == 1:
    axes = [axes]

for ax, e in zip(axes, sig_edges):
    ax.grid(False)
    col = f'{e}_{MEASURE}'
    sub = df_plot[[col, 'shaps_total', 'anhedonia_group']].dropna()
    x, y = sub['shaps_total'].to_numpy(), sub[col].to_numpy()

    # points colored by group
    for g, c in GROUP_COLORS.items():
        m = sub['anhedonia_group'] == g
        ax.scatter(x[m.to_numpy()], y[m.to_numpy()], s=28, alpha=0.65,
                   color=c, edgecolor='white', linewidth=0.5, label=g)

    # OLS fit line over the WHOLE sample
    b, a = np.polyfit(x, y, 1)                       # slope, intercept
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, a + b * xs, color='black', lw=1.8)

    r, p = pearsonr(x, y)
    ax.set_title(f'{e}', fontsize=12, fontweight='bold')
    ax.set_xlabel('SHAPS total')
    ax.set_ylabel(f'{e} {MEASURE} (P→S − S→P)')

    ax.xaxis.set_major_formatter(FormatStrFormatter('%.0f'))   # SHAPS as integers
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))   # one decimal
    # ax.text(0.04, 0.96, f'r = {r:+.3f}\np = {p:.4f}\nn = {len(sub)}',
    #         transform=ax.transAxes, va='top', ha='left', fontsize=10,
    #         bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='0.7', alpha=0.85))

# axes[-1].legend(title='Anhedonia', frameon=False, loc='lower right')
fig.suptitle(f'Edge asymmetry ({MEASURE}) vs SHAPS total — whole sample', y=1.02, fontsize=13)
fig.tight_layout()
plt.savefig('scatter_edge_asym_vs_shaps.png', dpi=600, bbox_inches='tight')
plt.show()


### with TEPS

In [ ]:
teps_items = pd.read_csv('/Users/proghani/Documents/personal/local_thesis_proj/TCP_thesis_project/phenotype/imputed/imputed_teps01.csv')

In [ ]:
# %% Scatter plots: per-edge asymmetry vs total TEPS (whole sample, both groups pooled)
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

# --- TEPS total (same construction as shaps_total: first col = ID, rest = items) ---
teps_items['teps_total'] = teps_items.iloc[:, 1:].sum(axis=1)
teps_id = teps_items.columns[0]
teps = teps_items[[teps_id, 'teps_total']].rename(columns={teps_id: 'subjectkey'})

MEASURE   = 'norm'
sig_edges = ['vmPFC_NAc', 'dlPFC_NAc', 'ACC_aPUT']
GROUP_COLORS = {'LOW': '#3AAFB9', 'HIGH': '#D65A7A'}

plot_cols = ['subjectkey', 'anhedonia_group'] + [f'{e}_{MEASURE}' for e in sig_edges]
df_plot = df_fs[plot_cols].merge(teps, on='subjectkey', how='inner')
print(f'Merged rows: {len(df_plot)}  (df_fs={len(df_fs)}, teps={len(teps)})')

fig, axes = plt.subplots(1, len(sig_edges), figsize=(5 * len(sig_edges), 4.5))
if len(sig_edges) == 1:
    axes = [axes]

for ax, e in zip(axes, sig_edges):
    col = f'{e}_{MEASURE}'
    sub = df_plot[[col, 'teps_total', 'anhedonia_group']].dropna()
    x, y = sub['teps_total'].to_numpy(), sub[col].to_numpy()

    for g, c in GROUP_COLORS.items():
        m = (sub['anhedonia_group'] == g).to_numpy()
        ax.scatter(x[m], y[m], s=28, alpha=0.65, color=c,
                   edgecolor='white', linewidth=0.5, label=g)

    # pooled OLS fit line over the whole sample
    b, a = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, a + b * xs, color='black', lw=1.8)

    r, p = pearsonr(x, y)
    ax.set_title(f'{e}', fontsize=12, fontweight='bold')
    ax.set_xlabel('TEPS total')
    ax.set_ylabel(f'{e} {MEASURE} (P→S − S→P)')
    ax.text(0.04, 0.96, f'r = {r:+.3f}\np = {p:.4f}',
            transform=ax.transAxes, va='top', ha='left', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='0.7', alpha=0.85))

axes[-1].legend(title='Anhedonia', frameon=False, loc='lower right')
fig.suptitle(f'Edge asymmetry ({MEASURE}) vs TEPS total — whole sample', y=1.02, fontsize=13)
fig.tight_layout()
# plt.savefig('scatter_edge_asym_vs_teps.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# %% Scatter plots: per-edge asymmetry vs TEPS SUBSCALES (whole sample, both groups pooled)
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter
from scipy.stats import pearsonr

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11})

# --- TEPS subscale definitions (column names = teps1..teps18) ---
ANTICIPATORY = [1, 4, 6, 8, 10, 11, 13, 15, 16, 18]
CONSUMMATORY = [2, 3, 5, 7, 9, 12, 14, 17]
SUBSCALES = {
    'Anticipatory': [f'teps{i}' for i in ANTICIPATORY],
    'Consummatory': [f'teps{i}' for i in CONSUMMATORY],
}

# sanity check: every item is used exactly once
assert sorted(ANTICIPATORY + CONSUMMATORY) == list(range(1, 19)), 'item assignment mismatch'

# build subscale totals (first col = subjectkey, same construction as teps_total)
teps_sub = teps_items[['subjectkey']].copy()
for name, cols in SUBSCALES.items():
    teps_sub[name] = teps_items[cols].sum(axis=1)

MEASURE      = 'norm'
sig_edges    = ['vmPFC_NAc', 'dlPFC_NAc', 'ACC_aPUT']
GROUP_COLORS = {'LOW': '#3AAFB9', 'HIGH': '#D65A7A'}
sub_names    = list(SUBSCALES.keys())

plot_cols = ['subjectkey', 'anhedonia_group'] + [f'{e}_{MEASURE}' for e in sig_edges]
df_plot = df_fs[plot_cols].merge(teps_sub, on='subjectkey', how='inner')
print(f'Merged rows: {len(df_plot)}  (df_fs={len(df_fs)}, teps={len(teps_sub)})')

fig, axes = plt.subplots(len(sub_names), len(sig_edges),
                         figsize=(5 * len(sig_edges), 4.5 * len(sub_names)),
                         squeeze=False)

for i, subscale in enumerate(sub_names):
    for j, e in enumerate(sig_edges):
        ax  = axes[i, j]
        ax.grid(False)
        col = f'{e}_{MEASURE}'
        sub = df_plot[[col, subscale, 'anhedonia_group']].dropna()
        x, y = sub[subscale].to_numpy(), sub[col].to_numpy()

        for g, c in GROUP_COLORS.items():
            m = (sub['anhedonia_group'] == g).to_numpy()
            ax.scatter(x[m], y[m], s=28, alpha=0.65, color=c,
                       edgecolor='white', linewidth=0.5, label=g)

        # pooled OLS fit line over the whole sample
        b, a = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 100)
        ax.plot(xs, a + b * xs, color='black', lw=1.8)

        r, p = pearsonr(x, y)
        ax.set_title(f'{e}  ·  {subscale}', fontsize=12, fontweight='bold')
        ax.set_xlabel(f'TEPS {subscale}')
        ax.set_ylabel(f'{e} {MEASURE} (P→S − S→P)')

        ax.xaxis.set_major_formatter(FormatStrFormatter('%.0f'))   # TEPS as integers
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))   # one decimal
        # ax.text(0.04, 0.96, f'r = {r:+.3f}\np = {p:.4f}',
        #         transform=ax.transAxes, va='top', ha='left', fontsize=10,
        #         bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='0.7', alpha=0.85))

# axes[0, -1].legend(title='Anhedonia', frameon=False, loc='lower right')
fig.suptitle(f'Edge asymmetry ({MEASURE}) vs TEPS subscales — whole sample', y=1.00, fontsize=13)
fig.tight_layout()
plt.savefig('scatter_edge_asym_vs_teps_subscales.png', dpi=600, bbox_inches='tight')
plt.show()
